# Un pipeline de aprendizaje automático con Prefect

## De qué trata este notebook

El primer notebook de esta carpeta presenta las piezas de Prefect una por una. Este
las usa juntas para construir lo que aparece en cualquier sistema de aprendizaje
automático en producción: un **pipeline de entrenamiento e inferencia** que descarga
datos, los valida, entrena un modelo, lo evalúa y produce predicciones, dejando
registro de cada paso.

El problema es predecir cuántas bicicletas públicas se alquilarán en Washington DC en
una hora dada, a partir de la fecha, la hora y el clima. Los datos son los mismos del
primer notebook (17.379 horas de 2011 y 2012); si ya se ejecutó, el CSV está en
`data/external/` y no se descarga de nuevo.

Todo el código vive en las celdas, y al final se vuelca a un archivo
`pipeline_bicis.py` que corre sin Jupyter. Ese archivo es el punto de partida para
programar el pipeline y para empaquetarlo en un contenedor.

## Bibliotecas que aparecen

| Biblioteca | Para qué se usa aquí |
|---|---|
| **Prefect** | orquestar: tasks, reintentos, caché, artifacts, trazabilidad |
| **pandas** | leer y manipular la tabla de datos (`DataFrame`) |
| **pandera** | declarar un *contrato de datos* —qué debe cumplir cada columna— y verificarlo |
| **scikit-learn** | entrenar el modelo (`HistGradientBoostingRegressor`) y calcular métricas |
| **pyarrow** | escribir las predicciones en formato Parquet (lo usa pandas por debajo) |
| **httpx** | descargar el respaldo de UCI si Hugging Face no responde |

## El pipeline

```
descargar ─► validar ─► construir_features ─► dividir_temporal ─┬─► entrenar ─► evaluar
                                                                 └────────────► predecir_lote
```

| Task | Qué garantiza | Pieza de Prefect |
|---|---|---|
| `descargar` | el dato llega aunque la red falle una vez | `retries` |
| `validar` | el dato cumple el contrato, o la ejecución se detiene **aquí** | `@task` |
| `construir_features` | solo entran al modelo las columnas permitidas | — |
| `dividir_temporal` | 2011 entrena, 2012 valida; nunca al azar | — |
| `entrenar` | no se reentrena si nada cambió | `cache_policy` |
| `evaluar` | la métrica queda publicada, no en un `print` | artifacts |
| `predecir_lote` | cada predicción sabe de qué ejecución salió | `prefect.runtime` |

## Requisitos

Los mismos del primer notebook: el servidor de Prefect en una terminal
(`uv run prefect server start`), `PREFECT_API_URL` configurada y la interfaz web
abierta en <http://127.0.0.1:4200>.

## 0. Comprobar el servidor

In [ ]:
import httpx
from prefect.settings import get_current_settings

api = get_current_settings().api.url
if not api:
    raise RuntimeError(
        "PREFECT_API_URL no esta configurada. En una terminal, una sola vez:\n"
        "    uv run prefect config set PREFECT_API_URL=http://127.0.0.1:4200/api"
    )
try:
    httpx.get(f"{api}/health", timeout=3).raise_for_status()
except httpx.HTTPError as exc:
    raise RuntimeError(
        f"El servidor de Prefect no responde en {api}. "
        "Hay que levantarlo en otra terminal con `uv run prefect server start` "
        "y volver a ejecutar esta celda."
    ) from exc

print("API de Prefect:", api)
print("Interfaz web:  ", api.removesuffix("/api"))

## 1. Imports y constantes

Todo lo que el archivo final necesita, junto. Dos listas merecen atención:

- `FEATURES` son las columnas que el modelo puede usar para predecir. No incluye
  `casuales` ni `registrados`: esas dos suman exactamente `total`, la variable a
  predecir, y un modelo que las viera "adivinaría" el resultado copiándolo. Ese error
  se llama **fuga de información** (*leakage*) y es de los más comunes en la práctica,
  porque el modelo parece excelente hasta que se usa con datos nuevos, donde esas
  columnas no existen todavía.
- `TARGET` es la variable a predecir.

In [ ]:
# Pipeline de entrenamiento e inferencia sobre el dataset de bicis, con Prefect.

# Generado por `02-pipeline-ml-con-prefect.ipynb`: el notebook construye estas tasks
# celda por celda y al final escribe este archivo. Para cambiarlo se edita el notebook
# (o `_generar_notebooks.py`) y se vuelve a ejecutar.
#
# Uso, desde esta carpeta:
#     uv run python pipeline_bicis.py

import io
import zipfile
from datetime import UTC, datetime, timedelta
from pathlib import Path

import httpx
import numpy as np
import pandas as pd
import pandera.pandas as pa
from prefect import flow, get_run_logger, task
from prefect.artifacts import create_markdown_artifact
from prefect.cache_policies import INPUTS, TASK_SOURCE
from prefect.runtime import flow_run
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [ ]:
# La raiz del repositorio se busca hacia arriba hasta encontrar pyproject.toml. Sin
# rutas absolutas: el notebook corre desde su carpeta y este archivo desde cualquier
# sitio (incluido /app dentro de un contenedor).
_inicio = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
RAIZ = _inicio
while not (RAIZ / "pyproject.toml").exists() and RAIZ.parent != RAIZ:
    RAIZ = RAIZ.parent

URL_HF = "https://huggingface.co/datasets/t22000t/bike-sharing-tabular/resolve/main/hour.csv"
URL_UCI = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
CACHE_CSV = RAIZ / "data" / "external" / "bicis-por-hora.csv"
PREDICCIONES = RAIZ / "data" / "processed" / "predicciones-bicis.parquet"

COLUMNAS = {
    "instant": "indice",
    "dteday": "fecha",
    "season": "estacion",
    "yr": "anio",
    "mnth": "mes",
    "hr": "hora",
    "holiday": "festivo",
    "weekday": "dia_semana",
    "workingday": "dia_laboral",
    "weathersit": "clima",
    "temp": "temperatura",
    "atemp": "sensacion_termica",
    "hum": "humedad",
    "windspeed": "viento",
    "casual": "casuales",
    "registered": "registrados",
    "cnt": "total",
}

TARGET = "total"
# `casuales` y `registrados` NO estan: suman exactamente `total`. Incluirlas seria
# predecir el target con el target (fuga de informacion), y el contrato lo verifica.
FEATURES = [
    "estacion",
    "anio",
    "mes",
    "hora",
    "festivo",
    "dia_semana",
    "dia_laboral",
    "clima",
    "temperatura",
    "sensacion_termica",
    "humedad",
    "viento",
]

## 2. `descargar`: la única task que habla con la red

Por eso es la única con reintentos. Y tiene su propia cache en disco (`CACHE_CSV`),
distinta de la de Prefect: la primera evita bajar 1,2 MB otra vez; la segunda, más
abajo, evita **entrenar** otra vez.

In [ ]:
@task(retries=3, retry_delay_seconds=[2, 5, 10], retry_jitter_factor=0.2)
def descargar(destino: Path = CACHE_CSV) -> pd.DataFrame:
    """Baja el CSV una vez y lo guarda en disco. Los reintentos cubren la red."""
    logger = get_run_logger()
    if destino.exists():
        logger.info("cache en disco: %s", destino.relative_to(RAIZ))
        return pd.read_csv(destino)
    try:
        crudo = pd.read_csv(URL_HF)
        logger.info("descargado de Hugging Face")
    except Exception as exc:  # cualquier fallo de red cae al respaldo
        logger.warning("Hugging Face no respondio (%s); usando UCI", type(exc).__name__)
        respuesta = httpx.get(URL_UCI, timeout=60, follow_redirects=True)
        respuesta.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(respuesta.content)) as zf:
            crudo = pd.read_csv(zf.open("hour.csv"))
    df = crudo.rename(columns=COLUMNAS)
    destino.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(destino, index=False)
    return df

## 3. `validar`: el contrato de datos, con pandera

**pandera** permite escribir, como código, lo que una tabla debe cumplir: qué columnas
tiene, de qué tipo, en qué rango. Ese conjunto de reglas es un **contrato de datos**.
El pipeline no *asume* que `hora` va de 0 a 23 ni que `total` es la suma de las otras
dos columnas: lo **verifica**, y si algo no se cumple, la ejecución falla en esta task
con un mensaje claro, y no dentro del entrenamiento con un error críptico.

Tres detalles del esquema:

- `pa.Column(int, pa.Check.in_range(0, 23))` declara tipo y rango de una columna.
- El check a nivel de tabla (`checks=[...]`) ve varias columnas a la vez; aquí
  verifica la igualdad que justifica excluir dos columnas de `FEATURES`.
- `lazy=True` al validar reporta **todos** los errores juntos, en vez de detenerse
  en el primero.

In [ ]:
CONTRATO = pa.DataFrameSchema(
    {
        "fecha": pa.Column(str),
        "anio": pa.Column(int, pa.Check.isin([0, 1])),
        "mes": pa.Column(int, pa.Check.in_range(1, 12)),
        "hora": pa.Column(int, pa.Check.in_range(0, 23)),
        "clima": pa.Column(int, pa.Check.in_range(1, 4)),
        "temperatura": pa.Column(float, pa.Check.in_range(0, 1)),
        "humedad": pa.Column(float, pa.Check.in_range(0, 1)),
        "viento": pa.Column(float, pa.Check.ge(0)),
        "casuales": pa.Column(int, pa.Check.ge(0)),
        "registrados": pa.Column(int, pa.Check.ge(0)),
        "total": pa.Column(int, pa.Check.ge(0)),
    },
    checks=[
        # La regla que justifica excluir dos columnas de FEATURES, verificada y no supuesta.
        pa.Check(lambda d: d["casuales"] + d["registrados"] == d["total"], name="total_es_la_suma"),
    ],
    strict=False,  # las columnas que no se nombran pasan sin revisarse
)


@task
def validar(df: pd.DataFrame) -> pd.DataFrame:
    """Detiene la ejecucion si los datos no cumplen el contrato. Reporta todos los errores."""
    validado = CONTRATO.validate(df, lazy=True)
    get_run_logger().info("contrato OK: %d filas, %d columnas", *validado.shape)
    return validado

Antes de seguir, el contrato en acción sobre datos rotos a propósito: una fila con
`hora = 25` y otra con un `total` que no es la suma. Las dos fallas se reportan en
una sola pasada. El check de tabla completa (`total_es_la_suma`) reporta la **fila**
entera —una entrada por columna—, por eso la salida se resume por check en vez de
imprimir `failure_cases` crudo.

In [ ]:
# Solo pandas, sin Prefect: la task usa get_run_logger() y fuera de una ejecucion no hay logger.
muestra = pd.read_csv(CACHE_CSV) if CACHE_CSV.exists() else pd.read_csv(URL_HF).rename(columns=COLUMNAS)
roto = muestra.head(100).copy()
roto.loc[0, "hora"] = 25
roto.loc[1, "total"] = roto.loc[1, "total"] + 1
try:
    CONTRATO.validate(roto, lazy=True)
except pa.errors.SchemaErrors as exc:
    fallos = exc.failure_cases
    print("filas que fallan, por check:")
    print(fallos.groupby("check")["index"].nunique().to_string(), "\n")
    print("detalle de los checks de columna:")
    columna = fallos[fallos["schema_context"] == "Column"]
    print(columna[["column", "check", "index", "failure_case"]].to_string(index=False))

## 4. Features y división temporal

Dos tasks pequeñas y una regla que no se negocia: **la división entre entrenamiento y
validación es por tiempo**. Lo habitual en un tutorial es `train_test_split`, que
reparte las filas al azar. Con datos ordenados en el tiempo eso es un error: la hora
14 del 3 de marzo caería en entrenamiento y la hora 15 del mismo día en validación, y
el modelo "sabría" lo que pasó una hora antes de lo que tiene que predecir. La
métrica saldría optimista y el modelo decepcionaría al usarse.

Con 2011 para entrenar y 2012 para validar, el modelo se evalúa como se va a usar:
prediciendo lo que aún no ha pasado.

In [ ]:
@task
def construir_features(df: pd.DataFrame) -> pd.DataFrame:
    """Se queda con FEATURES + TARGET + fecha. Lo demas no entra al modelo."""
    return df[["fecha", *FEATURES, TARGET]].copy()


@task
def dividir_temporal(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """2011 entrena, 2012 valida. Nunca un split aleatorio: es una serie en el tiempo."""
    entrena = df[df["anio"] == 0]
    valida = df[df["anio"] == 1]
    get_run_logger().info(
        "entrena %d filas (2011) | valida %d filas (2012)", len(entrena), len(valida)
    )
    return entrena, valida

## 5. `entrenar`: cacheado, y por qué es seguro

El modelo es un `HistGradientBoostingRegressor` de scikit-learn: un conjunto de
árboles de decisión que se construyen uno tras otro, cada uno corrigiendo los errores
del anterior (*gradient boosting*). Es rápido, funciona bien con datos tabulares sin
mucho preprocesamiento y tiene dos hiperparámetros principales: cuántos árboles
(`max_iter`) y cuánto corrige cada uno (`learning_rate`).

Es la task cara (segundos aquí, horas en un caso real), y por eso lleva caché.
`INPUTS + TASK_SOURCE` significa: mismo DataFrame de entrenamiento, mismos
hiperparámetros y mismo código → mismo modelo, leído de disco sin entrenar. Si
cualquiera de los tres cambia, se reentrena solo.

La condición para que esto no sea una trampa es `random_state=42`: fija la semilla
aleatoria del algoritmo, de modo que dos entrenamientos con las mismas entradas
producen el mismo modelo. Sin ella, cachear sería reutilizar un modelo que nadie
evaluó.

In [ ]:
@task(cache_policy=INPUTS + TASK_SOURCE, cache_expiration=timedelta(days=1))
def entrenar(entrena: pd.DataFrame, n_iteraciones: int, tasa_aprendizaje: float):
    """Entrena el regresor. Cacheado: mismos datos + mismos hiperparametros = mismo modelo.

    Es seguro cachear porque `random_state` esta fijo y el DataFrame de entrada es parte
    de la clave: si los datos cambian, se reentrena solo. Sin `random_state`, cachear
    seria servir un modelo que nadie evaluo.
    """
    modelo = HistGradientBoostingRegressor(
        max_iter=n_iteraciones, learning_rate=tasa_aprendizaje, random_state=42
    )
    modelo.fit(entrena[FEATURES], entrena[TARGET])
    return modelo

## 6. `evaluar` y `predecir_lote`

`evaluar` calcula dos métricas de error sobre 2012:

- **RMSE** (raíz del error cuadrático medio): penaliza más los errores grandes. Está
  en las mismas unidades que el target, aquí bicicletas por hora.
- **MAE** (error absoluto medio): el error típico, sin castigar los extremos.

Y compara contra un **baseline**: predecir siempre el promedio de 2011. Si el modelo
no le gana a esa regla trivial, no aprendió nada, por bien que suene su métrica sola.
El resultado va a un artifact con clave fija, `bicis-metricas`, así la interfaz
muestra su historia ejecución tras ejecución.

`predecir_lote` genera las predicciones de un mes completo y escribe cada fila con el
identificador y el nombre de la ejecución que la produjo (`prefect.runtime.flow_run`)
y la hora en que se generó. Es **trazabilidad por fila**: dada cualquier predicción,
se puede llegar a la ejecución, sus logs, sus parámetros y su modelo. Las predicciones
se guardan en **Parquet**, un formato de columnas comprimido, más compacto y rápido
que CSV y que conserva los tipos de dato.

In [ ]:
@task
def evaluar(modelo, entrena: pd.DataFrame, valida: pd.DataFrame) -> dict:
    """RMSE y MAE del modelo contra el baseline de predecir la media de 2011."""
    y = valida[TARGET]
    baseline = np.full(len(valida), entrena[TARGET].mean())
    pred = modelo.predict(valida[FEATURES])
    metricas = {
        "rmse_baseline": round(float(root_mean_squared_error(y, baseline)), 1),
        "rmse_modelo": round(float(root_mean_squared_error(y, pred)), 1),
        "mae_modelo": round(float(mean_absolute_error(y, pred)), 1),
    }
    filas = "\n".join(f"| {k} | {v} |" for k, v in metricas.items())
    create_markdown_artifact(
        key="bicis-metricas",
        markdown=f"# Metricas en 2012\n\n| metrica | valor |\n|---|---|\n{filas}\n",
        description="Validacion temporal: entrena 2011, evalua 2012.",
    )
    get_run_logger().info("metricas: %s", metricas)
    return metricas

In [ ]:
@task
def predecir_lote(modelo, valida: pd.DataFrame, mes: int, destino: Path = PREDICCIONES) -> Path:
    """Predice un mes completo y guarda cada fila con el id de la ejecucion que la produjo."""
    lote = valida[valida["mes"] == mes].copy()
    lote["prediccion"] = modelo.predict(lote[FEATURES]).round(0)
    # Trazabilidad por fila: de que ejecucion salio cada prediccion y cuando.
    lote["flow_run_id"] = flow_run.id
    lote["flow_run_name"] = flow_run.name
    lote["generado_en"] = datetime.now(UTC).isoformat(timespec="seconds")
    destino.parent.mkdir(parents=True, exist_ok=True)
    lote.to_parquet(destino, index=False)
    get_run_logger().info("%d predicciones en %s", len(lote), destino.relative_to(RAIZ))
    return destino

## 7. El flow: siete tasks, un grafo

El orden lo dan los datos: `validar` necesita lo que devuelve `descargar`, `entrenar`
necesita `entrena`, y así sucesivamente. No hay una sola línea que diga "esto va
después de aquello".

In [ ]:
@flow(log_prints=True)
def pipeline_bicis(
    n_iteraciones: int = 300, tasa_aprendizaje: float = 0.1, mes_a_predecir: int = 12
) -> dict:
    """Descarga -> valida -> features -> split temporal -> entrena -> evalua -> predice."""
    crudo = descargar()
    limpio = validar(crudo)
    tabla = construir_features(limpio)
    entrena, valida = dividir_temporal(tabla)
    modelo = entrenar(entrena, n_iteraciones, tasa_aprendizaje)
    metricas = evaluar(modelo, entrena, valida)
    predecir_lote(modelo, valida, mes_a_predecir)
    return metricas

In [ ]:
import time

inicio = time.perf_counter()
metricas = pipeline_bicis()
print(f"\n{time.perf_counter() - inicio:.1f}s | {metricas}")

El modelo supera al baseline con margen amplio: un RMSE cercano a 125 bicicletas por
hora frente a unas 228 del promedio fijo. En la interfaz: la ejecución con sus siete
tasks, el artifact `bicis-metricas`, y en el log de `predecir_lote` la ruta del
Parquet.

Ahora la **segunda ejecución**, sin cambiar nada:

In [ ]:
inicio = time.perf_counter()
pipeline_bicis()
print(f"\n{time.perf_counter() - inicio:.1f}s  <- `entrenar` en estado Cached")

Más rápida, y en la interfaz `entrenar` aparece como **Cached**. Con otros
hiperparámetros —`pipeline_bicis(n_iteraciones=100)`— vuelve a entrenar, porque
cambió una entrada.

Si este notebook ya se había ejecutado antes, la **primera** ejecución también salió
rápida y con `entrenar` en `Cached`: la caché está en disco (`~/.prefect/storage/`),
no en el kernel, y dura lo que diga `cache_expiration` (aquí, un día). Es el
comportamiento correcto para un pipeline, aunque le quite contraste a la demostración.

Las predicciones, con su procedencia en cada fila:

In [ ]:
predicciones = pd.read_parquet(PREDICCIONES)
print(predicciones.shape, "| ejecucion:", predicciones["flow_run_name"].iloc[0])
predicciones[["fecha", "hora", "total", "prediccion", "flow_run_id", "generado_en"]].head(8)

## 8. Del notebook al archivo

Todo lo anterior, junto, en `pipeline_bicis.py`. Es **el mismo código** de las celdas;
el notebook lo escribe para que no existan dos versiones que mantener. A partir de
aquí, el pipeline ya no necesita Jupyter.

In [ ]:
CODIGO_PIPELINE = r'''
"""Pipeline de entrenamiento e inferencia sobre el dataset de bicis, con Prefect.

Generado por `02-pipeline-ml-con-prefect.ipynb`: el notebook construye estas tasks
celda por celda y al final escribe este archivo. Para cambiarlo se edita el notebook
(o `_generar_notebooks.py`) y se vuelve a ejecutar.

Uso, desde esta carpeta:
    uv run python pipeline_bicis.py
"""

import io
import zipfile
from datetime import UTC, datetime, timedelta
from pathlib import Path

import httpx
import numpy as np
import pandas as pd
import pandera.pandas as pa
from prefect import flow, get_run_logger, task
from prefect.artifacts import create_markdown_artifact
from prefect.cache_policies import INPUTS, TASK_SOURCE
from prefect.runtime import flow_run
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# La raiz del repositorio se busca hacia arriba hasta encontrar pyproject.toml. Sin
# rutas absolutas: el notebook corre desde su carpeta y este archivo desde cualquier
# sitio (incluido /app dentro de un contenedor).
_inicio = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
RAIZ = _inicio
while not (RAIZ / "pyproject.toml").exists() and RAIZ.parent != RAIZ:
    RAIZ = RAIZ.parent

URL_HF = "https://huggingface.co/datasets/t22000t/bike-sharing-tabular/resolve/main/hour.csv"
URL_UCI = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
CACHE_CSV = RAIZ / "data" / "external" / "bicis-por-hora.csv"
PREDICCIONES = RAIZ / "data" / "processed" / "predicciones-bicis.parquet"

COLUMNAS = {
    "instant": "indice",
    "dteday": "fecha",
    "season": "estacion",
    "yr": "anio",
    "mnth": "mes",
    "hr": "hora",
    "holiday": "festivo",
    "weekday": "dia_semana",
    "workingday": "dia_laboral",
    "weathersit": "clima",
    "temp": "temperatura",
    "atemp": "sensacion_termica",
    "hum": "humedad",
    "windspeed": "viento",
    "casual": "casuales",
    "registered": "registrados",
    "cnt": "total",
}

TARGET = "total"
# `casuales` y `registrados` NO estan: suman exactamente `total`. Incluirlas seria
# predecir el target con el target (fuga de informacion), y el contrato lo verifica.
FEATURES = [
    "estacion",
    "anio",
    "mes",
    "hora",
    "festivo",
    "dia_semana",
    "dia_laboral",
    "clima",
    "temperatura",
    "sensacion_termica",
    "humedad",
    "viento",
]


@task(retries=3, retry_delay_seconds=[2, 5, 10], retry_jitter_factor=0.2)
def descargar(destino: Path = CACHE_CSV) -> pd.DataFrame:
    """Baja el CSV una vez y lo guarda en disco. Los reintentos cubren la red."""
    logger = get_run_logger()
    if destino.exists():
        logger.info("cache en disco: %s", destino.relative_to(RAIZ))
        return pd.read_csv(destino)
    try:
        crudo = pd.read_csv(URL_HF)
        logger.info("descargado de Hugging Face")
    except Exception as exc:  # cualquier fallo de red cae al respaldo
        logger.warning("Hugging Face no respondio (%s); usando UCI", type(exc).__name__)
        respuesta = httpx.get(URL_UCI, timeout=60, follow_redirects=True)
        respuesta.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(respuesta.content)) as zf:
            crudo = pd.read_csv(zf.open("hour.csv"))
    df = crudo.rename(columns=COLUMNAS)
    destino.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(destino, index=False)
    return df


CONTRATO = pa.DataFrameSchema(
    {
        "fecha": pa.Column(str),
        "anio": pa.Column(int, pa.Check.isin([0, 1])),
        "mes": pa.Column(int, pa.Check.in_range(1, 12)),
        "hora": pa.Column(int, pa.Check.in_range(0, 23)),
        "clima": pa.Column(int, pa.Check.in_range(1, 4)),
        "temperatura": pa.Column(float, pa.Check.in_range(0, 1)),
        "humedad": pa.Column(float, pa.Check.in_range(0, 1)),
        "viento": pa.Column(float, pa.Check.ge(0)),
        "casuales": pa.Column(int, pa.Check.ge(0)),
        "registrados": pa.Column(int, pa.Check.ge(0)),
        "total": pa.Column(int, pa.Check.ge(0)),
    },
    checks=[
        # La regla que justifica excluir dos columnas de FEATURES, verificada y no supuesta.
        pa.Check(lambda d: d["casuales"] + d["registrados"] == d["total"], name="total_es_la_suma"),
    ],
    strict=False,  # las columnas que no se nombran pasan sin revisarse
)


@task
def validar(df: pd.DataFrame) -> pd.DataFrame:
    """Detiene la ejecucion si los datos no cumplen el contrato. Reporta todos los errores."""
    validado = CONTRATO.validate(df, lazy=True)
    get_run_logger().info("contrato OK: %d filas, %d columnas", *validado.shape)
    return validado


@task
def construir_features(df: pd.DataFrame) -> pd.DataFrame:
    """Se queda con FEATURES + TARGET + fecha. Lo demas no entra al modelo."""
    return df[["fecha", *FEATURES, TARGET]].copy()


@task
def dividir_temporal(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """2011 entrena, 2012 valida. Nunca un split aleatorio: es una serie en el tiempo."""
    entrena = df[df["anio"] == 0]
    valida = df[df["anio"] == 1]
    get_run_logger().info(
        "entrena %d filas (2011) | valida %d filas (2012)", len(entrena), len(valida)
    )
    return entrena, valida


@task(cache_policy=INPUTS + TASK_SOURCE, cache_expiration=timedelta(days=1))
def entrenar(entrena: pd.DataFrame, n_iteraciones: int, tasa_aprendizaje: float):
    """Entrena el regresor. Cacheado: mismos datos + mismos hiperparametros = mismo modelo.

    Es seguro cachear porque `random_state` esta fijo y el DataFrame de entrada es parte
    de la clave: si los datos cambian, se reentrena solo. Sin `random_state`, cachear
    seria servir un modelo que nadie evaluo.
    """
    modelo = HistGradientBoostingRegressor(
        max_iter=n_iteraciones, learning_rate=tasa_aprendizaje, random_state=42
    )
    modelo.fit(entrena[FEATURES], entrena[TARGET])
    return modelo


@task
def evaluar(modelo, entrena: pd.DataFrame, valida: pd.DataFrame) -> dict:
    """RMSE y MAE del modelo contra el baseline de predecir la media de 2011."""
    y = valida[TARGET]
    baseline = np.full(len(valida), entrena[TARGET].mean())
    pred = modelo.predict(valida[FEATURES])
    metricas = {
        "rmse_baseline": round(float(root_mean_squared_error(y, baseline)), 1),
        "rmse_modelo": round(float(root_mean_squared_error(y, pred)), 1),
        "mae_modelo": round(float(mean_absolute_error(y, pred)), 1),
    }
    filas = "\n".join(f"| {k} | {v} |" for k, v in metricas.items())
    create_markdown_artifact(
        key="bicis-metricas",
        markdown=f"# Metricas en 2012\n\n| metrica | valor |\n|---|---|\n{filas}\n",
        description="Validacion temporal: entrena 2011, evalua 2012.",
    )
    get_run_logger().info("metricas: %s", metricas)
    return metricas


@task
def predecir_lote(modelo, valida: pd.DataFrame, mes: int, destino: Path = PREDICCIONES) -> Path:
    """Predice un mes completo y guarda cada fila con el id de la ejecucion que la produjo."""
    lote = valida[valida["mes"] == mes].copy()
    lote["prediccion"] = modelo.predict(lote[FEATURES]).round(0)
    # Trazabilidad por fila: de que ejecucion salio cada prediccion y cuando.
    lote["flow_run_id"] = flow_run.id
    lote["flow_run_name"] = flow_run.name
    lote["generado_en"] = datetime.now(UTC).isoformat(timespec="seconds")
    destino.parent.mkdir(parents=True, exist_ok=True)
    lote.to_parquet(destino, index=False)
    get_run_logger().info("%d predicciones en %s", len(lote), destino.relative_to(RAIZ))
    return destino


@flow(log_prints=True)
def pipeline_bicis(
    n_iteraciones: int = 300, tasa_aprendizaje: float = 0.1, mes_a_predecir: int = 12
) -> dict:
    """Descarga -> valida -> features -> split temporal -> entrena -> evalua -> predice."""
    crudo = descargar()
    limpio = validar(crudo)
    tabla = construir_features(limpio)
    entrena, valida = dividir_temporal(tabla)
    modelo = entrenar(entrena, n_iteraciones, tasa_aprendizaje)
    metricas = evaluar(modelo, entrena, valida)
    predecir_lote(modelo, valida, mes_a_predecir)
    return metricas


if __name__ == "__main__":
    print(pipeline_bicis())
'''

destino = Path("pipeline_bicis.py")
destino.write_text(CODIGO_PIPELINE.lstrip("\n"), encoding="utf-8")
print(destino.resolve().relative_to(RAIZ), "-", len(destino.read_text().splitlines()), "lineas")

La prueba de que es independiente: ejecutarlo como script, en **otro proceso**, con el
mismo intérprete de Python de este entorno. `subprocess` es el módulo estándar para
lanzar programas externos desde Python.

In [ ]:
import subprocess
import sys

salida = subprocess.run(
    [sys.executable, "pipeline_bicis.py"], capture_output=True, text=True, check=True
)
print(salida.stdout.strip().splitlines()[-1])

## 9. Programarlo: lo que sigue es terminal

Un pipeline que solo corre cuando alguien ejecuta una celda no resuelve el problema
original. Prefect ofrece `serve()`: deja el flow **escuchando** en un proceso y lo
ejecuta según un horario (`cron`) o cuando alguien lo lanza desde la interfaz, con los
parámetros que se le pasen. Ese proceso no termina, así que no se ejecuta desde el
notebook (bloquearía el kernel), sino desde una terminal en esta carpeta:

```bash
# Ejecutarlo una vez
uv run python pipeline_bicis.py

# Servirlo con un horario: todos los dias a las 6:00. Queda en Deployments en la interfaz.
uv run python -c "from pipeline_bicis import pipeline_bicis; pipeline_bicis.serve(name='bicis-diario', cron='0 6 * * *')"
```

La expresión `0 6 * * *` es sintaxis **cron**, el formato estándar de Unix para
horarios: minuto 0, hora 6, cualquier día, mes y día de la semana. Con eso, el
pipeline corre cada mañana sin nadie delante, y desde la interfaz se puede lanzar a
mano con otros parámetros —`n_iteraciones`, `mes_a_predecir`— que Prefect valida antes
de ejecutar.

## Resumen

Siete tasks que juntas forman un pipeline completo:

| Etapa | Concepto | Cómo se protege |
|---|---|---|
| descarga | fuentes externas fallan | `retries` con backoff y una fuente de respaldo |
| validación | los datos pueden venir mal | contrato de datos con pandera, antes de tocar el modelo |
| features | fuga de información | `FEATURES` excluye lo que "es" el target, y el contrato lo verifica |
| división | series de tiempo | por año, nunca al azar |
| entrenamiento | trabajo caro y repetido | caché por entradas + código, con semilla fija |
| evaluación | métricas sin referencia | baseline explícito y artifact con historia |
| inferencia | predicciones sin origen | id de la ejecución en cada fila |

Siguiente: [`../../s05-deployment/notebooks/01-el-flow-en-un-contenedor.ipynb`](../../s05-deployment/notebooks/01-el-flow-en-un-contenedor.ipynb),
donde este mismo `pipeline_bicis.py` se empaqueta en una imagen de Docker y se
ejecuta dentro de un contenedor.